In [2]:
# Install all required packages for LangGraph and LangChain
!pip install -q langgraph langchain langchain-groq python-dotenv langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [7]:
import os
from google.colab import userdata

# This tells Colab: Look for the secret named "GROQ_API_KEY"
# and save its hidden value into the system environment.
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [8]:
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

# 1. Define the State
class PipelineState(TypedDict):
    raw_input : str
    edited_text : str
    script_text : str
    final_output : str

# Initialize the LLM (using the Groq API key loaded in Step 3)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

# 2. Define the Nodes
def editor_node(state: PipelineState) -> dict:
    """Stage 1: Cleans up grammar, removes typos, and refines the tone."""
    print("\n--- [Stage 1] Executing Editor Node ---")
    prompt = (
        "You are an expert copyeditor. Clean up the following raw text. "
        "Fix any grammatical errors, spelling mistakes, and smooth out the transition flow "
        "while keeping the core message intact. Return only the edited text.\n\n"
        f"Text:\n{state['raw_input']}"
    )
    response = llm.invoke(prompt)
    return {"edited_text": response.content.strip()}

def scriptwriter_node(state: PipelineState) -> dict:
    """Stage 2: Formats the clean text into an engaging video script style."""
    print("\n--- [Stage 2] Executing Scriptwriter Node ---")
    prompt = (
        "You are a charismatic YouTube content creator. Take this edited text and transform "
        "it into a highly engaging, punchy, conversational video script hook. Make it sound "
        "like a real person speaking passionately. Return only the script content.\n\n"
        f"Edited Text:\n{state['edited_text']}"
    )
    response = llm.invoke(prompt)
    return {"script_text": response.content.strip()}

def translator_node(state: PipelineState) -> dict:
    """Stage 3: Translates the script into natural flowing Hinglish."""
    print("\n--- [Stage 3] Executing Hinglish Translator Node ---")
    prompt = (
        "You are an expert content localizer for the Indian market. Take the following script "
        "and convert it into natural, flowing 'Hinglish'. Do not simply translate it sentence-by-sentence "
        "or repeat information. Alternating comfortably between Hindi and English phrases just like "
        "an intellectual tech educator would speak naturally on a live stream. Keep the energy high! "
        "Return only the final Hinglish text.\n\n"
        f"Script:\n{state['script_text']}"
    )
    response = llm.invoke(prompt)
    return {"final_output": response.content.strip()}

# 3. Create and Build the Graph
graph = StateGraph(PipelineState)

# Add the nodes
graph.add_node("editor", editor_node)
graph.add_node("scriptwriter", scriptwriter_node)
graph.add_node("translator", translator_node)

# Add sequential edges
graph.add_edge(START, "editor")
graph.add_edge("editor", "scriptwriter")
graph.add_edge("scriptwriter", "translator")
graph.add_edge("translator", END)

# Compile the graph
app = graph.compile()

# 4. Execute the Graph
print("Starting the LangGraph Workflow...")
result = app.invoke({
    "raw_input": "AI agents are the future of tech. They can think, plan, and act on their own. LangGraph helps you build these agents with proper control and memory."
})

# Output Results
print("\n================ FINAL RESULT ================\n")
print(result['final_output'])

Starting the LangGraph Workflow...

--- [Stage 1] Executing Editor Node ---

--- [Stage 2] Executing Scriptwriter Node ---

--- [Stage 3] Executing Hinglish Translator Node ---

================ FINAL RESULT ================

Arre yaar, aaj hum baat karte hain tech ki future ki - main is cheez mein completely obsessed hoon! Hum AI agents ki revolution ke threshold par khade hain, jo ki literally soch sakte hain, plan kar sakte hain, aur apne aap action le sakte hain. I mean, possibilities toh aap soch bhi nahi sakte, na? Aur best part yeh hai ki humare paas tools hai jo ise possible banate hain. Main aapko introduce karna chahta hoon LangGraph se, jo ki ek game-changing platform hai jo aapko in autonomous agents ko ground up build karne ki permission deti hai, aapko total control deti hai, aur unhein jo memory chahiye hoti hai wo bhi provide karti hai. Yeh toh jaise ki aapke haath mein ek superpower hai, yaar!
